# Attention Pattern Dynamics

Analyze how attention patterns change: position dependence, context
sensitivity, entropy profiles, distance distributions, and pattern rank.

## Why This Matters

Attention pattern dynamics reveals how heads route information between positions. Understanding attention mechanics is central to reverse-engineering transformer circuits, as attention patterns determine what information each component can access and transform.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.attention_pattern_dynamics import (
    position_dependent_attention, attention_shift_between_contexts,
    attention_entropy_profile, attention_distance_profile,
    attention_pattern_rank,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
tokens_b = jnp.array([5, 20, 35, 50, 65, 80, 95])
print('Model ready')

## Position-Dependent Attention

How does the attention pattern vary by query position?

In [ ]:
result = position_dependent_attention(model, tokens, layer=0, head=0)
print(f"Entropy trend: {result['entropy_trend']:.4f}")
print(f"Mean entropy: {result['mean_entropy']:.4f}\n")
for p in result['per_position']:
    self_tag = '← SELF' if p['attends_self'] else ''
    print(f"  Pos {p['position']}: entropy={p['entropy']:.4f}, "
          f"max={p['max_attention']:.4f} (src={p['max_source']}), "
          f"sources={p['n_attended_sources']} {self_tag}")

## Context Sensitivity

How much does the pattern change between different inputs?

In [ ]:
result = attention_shift_between_contexts(model, tokens, tokens_b, layer=0, head=0)
sens = 'SENSITIVE' if result['is_context_sensitive'] else 'stable'
print(f"Mean JS divergence: {result['mean_js_divergence']:.4f} [{sens}]\n")
for p in result['per_position']:
    print(f"  Pos {p['position']}: JS={p['js_divergence']:.4f}, "
          f"entropy_change={p['entropy_change']:+.4f}")

## Entropy Profile

Map of sharp vs diffuse heads across all layers.

In [ ]:
result = attention_entropy_profile(model, tokens)
print(f"Sharp: {result['n_sharp_heads']}, Diffuse: {result['n_diffuse_heads']}")
print(f"Mean entropy: {result['mean_entropy']:.4f}\n")
for h in result['per_head']:
    tag = 'SHARP' if h['is_sharp'] else 'diffuse'
    print(f"  L{h['layer']}H{h['head']}: entropy={h['mean_entropy']:.4f} "
          f"(min={h['min_entropy']:.4f}, max={h['max_entropy']:.4f}) [{tag}]")

## Distance Profile

What distances does this head attend to?

In [ ]:
result = attention_distance_profile(model, tokens, layer=0, head=0)
local = 'LOCAL' if result['is_local'] else 'global'
print(f"Mean distance: {result['mean_attention_distance']:.4f} [{local}]\n")
for p in result['per_position']:
    print(f"  Pos {p['position']}: mean_dist={p['mean_distance']:.4f}, "
          f"relative={p['relative_distance']:.4f}")

## Pattern Rank

How complex is the attention pattern?

In [ ]:
result = attention_pattern_rank(model, tokens, layer=0, head=0)
rank_tag = 'LOW-RANK' if result['is_low_rank'] else 'full-rank'
print(f"Effective rank: {result['effective_rank']:.2f} / {result['max_rank']}")
print(f"Utilization: {result['rank_utilization']:.2%} [{rank_tag}]")
print(f"Rank for 90%: {result['rank_90_pct']}")
print(f"Top singular value: {result['top_singular_value']:.4f}")